# Running this experiment

See README.md for setup, required inputs, and limitations. Run cells in order from a fresh kernel. Saved outputs have been cleared; historical metrics are not newly verified results. Environment installation and external training commands are manual setup steps.

In [ ]:
from project_utils import project_path

# Override these paths for your local datasets. PROJECT_DATA_DIR changes their base.
INPUT_PATHS = {
    "TRAIN_CSV": project_path("train.csv"),
    "TEST_CSV": project_path("test.csv"),
    "GENDER_SUBMISSION_CSV": project_path("gender_submission.csv"),
    "CATBOOST_SUBMISSION_CSV": project_path("catboost_submission.csv"),
}

## Import dependencies


In [ ]:
# Import dependencies
pass  # Environment setup moved to README; use local inputs.

# Import Python libraries
import time, datetime

# Import data manipulation libraries
import matplotlib.pyplot as plt
import missingno  # library for visualization of missing data
import seaborn as sns

plt.style.use("seaborn-whitegrid")
import pandas as pd
import numpy as np

# Import preprocessing dependencies

# Import machine learning dependencies
from sklearn import metrics
from sklearn.svm import LinearSVC
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier

## Explore overall dataset

In [ ]:
pass  # Environment setup moved to README; use local inputs.
pass  # Environment setup moved to README; use local inputs.

In [ ]:
# Import training and test data
train = pd.read_csv(INPUT_PATHS["TRAIN_CSV"])
test = pd.read_csv(INPUT_PATHS["TEST_CSV"])
gender_submission = pd.read_csv(INPUT_PATHS["GENDER_SUBMISSION_CSV"])

In [ ]:
# view the training data
train.head(10)

In [ ]:
# Let's check the histogram of the ages of the passengers
train.Age.plot.hist()

In [ ]:
# view the test data
test.head(10)

In [ ]:
test.Age.plot.hist()

In [ ]:
# check the length of train and test set
len(train), len(test)

### Data description

**Survival:** 0 = No, 1 = Yes

**pclass:** 1=1st class, 2=2nd class, 3=3rd class

**sex:** Sex

**Age:** Age in years

**sibsp:** Number of siblings or spouses aboard the Titanic

**parch:** Number of parents or children aboard the Titanic

**ticket:** Ticket number

**fare:** Passenger fare

**cabin:** Cabin number

**embarked:** Port of embarkation, C=Cherbourg, Q=Queenstown, S=Southampton

In [ ]:
# get the statistics of the training data
train.describe()

In [ ]:
# plot graphic of missing values
missingno.matrix(train, figsize=(30, 10))

Cabin and Age column has missing values

In [ ]:
train.isnull().sum()  # shows the number of missing values in each column
# finds null values of each column and adds them up

Let's create two new dataframes. One for exploring discretized continuous variables (continuous variables that have been sorted into some kind of category) and another for exploring continuous values

In [ ]:
# check the datatype of each column
train.dtypes

## Explore each feature individually

In [ ]:
df_bin = pd.DataFrame()  # for descredised continuous variables
df_con = pd.DataFrame()  # for continuous variables

### Target feature: Survived (predicted label)

In [ ]:
# how many people survived
fig = plt.figure(figsize=(20, 1))
sns.countplot(
    y="Survived", data=train
)  # countplot is a kind of histogram that shows the occurances of certain categorical values
print(train.Survived.value_counts())

In [ ]:
# Let's add this to our subset dataframe
df_bin["Survived"] = train["Survived"]
df_con["Survived"] = train["Survived"]

In [ ]:
df_bin.head()

In [ ]:
df_con.head()

### Feature: Pclass

In [ ]:
sns.distplot(train.Pclass)  # distribution plot (plots distri ution and range of values)

In [ ]:
df_bin["Pclass"] = train["Pclass"]
df_con["Pclass"] = train["Pclass"]

In [ ]:
df_bin.head()

### Feature: Name

In [ ]:
# How many unique kinds of Ticket are there?
len(train.Name.unique())

All names are unique. That's too many unique values. So we will not add this column to our subset dataframe.

### Feature: Sex

In [ ]:
# Let's view the distribution of the sexes
plt.figure(figsize=(20, 5))
sns.countplot(y="Sex", data=train)

In [ ]:
# add sex to subset dataframes
df_bin["Sex"] = train["Sex"]
df_bin["Sex"] = np.where(df_bin["Sex"] == "female", 1, 0)  # change sex to 0 for male and 1 female

df_con["Sex"] = train["Sex"]

New Function: np.where (look into its documentation)

In [ ]:
df_bin.head()

In [ ]:
# Let's see how the Sex variable looks commpared to the Survived variable
# We can compare them because they both possess binary values
fig = plt.figure(figsize=(10, 10))
sns.distplot(df_bin.loc[df_bin["Survived"] == 1]["Sex"], kde_kws={"label": "Survived"})
sns.distplot(df_bin.loc[df_bin["Survived"] == 0]["Sex"], kde_kws={"label": "Did not survive"});

New Function: sns.distplot (look into its documentation)




### Feature: Age

In [ ]:
train.Age.isnull().sum()  # total missing values in Age column

Function to create count and distribution visualizations (Please check countplot and distplot documentations)

In [ ]:
from experiment_kaggle_titanic_competition_submission import plot_count_dist

### Feature: SibSp


In [ ]:
# What values are there
train.SibSp.value_counts()

In [ ]:
# Add Sibsp to subset dataframes
df_bin["SibSp"] = train["SibSp"]
df_con["SibSp"] = train["SibSp"]

In [ ]:
# Visualize the counts of SibSp and the distribution of the values against survived
plot_count_dist(
    train,
    bin_df=df_bin,
    label_column="Survived",
    target_column="SibSp",
    figsize=(20, 10),
)

### Feature: Parch


In [ ]:
# What values are there
train.Parch.value_counts()

In [ ]:
# Add Parch to subset dataframes
df_bin["Parch"] = train["Parch"]
df_con["Parch"] = train["Parch"]

In [ ]:
# Visualise the counts of Parch and the distribution of the values
# against Survived
plot_count_dist(
    train,
    bin_df=df_bin,
    label_column="Survived",
    target_column="Parch",
    figsize=(20, 10),
)

### Feature: Ticket

In [ ]:
# How many kinds of ticket are there?
sns.countplot(y="Ticket", data=train);

In [ ]:
# How many kinds of ticket are there?
train.Ticket.value_counts()

In [ ]:
# How many unique kinds of Ticket are there?
len(train.Ticket.unique())

681 unique values is too many for now. So we won't use Ticket in our subset dataframes.

### Feature: Fare



In [ ]:
# How many unique kinds of Fare are there?
len(train.Fare.unique())

In [ ]:
# What kind of variable is Fare?
train.Fare.dtype

Because Fare is a float (number) let's add it as it is to our continuous sub dataframe but to add it to our categorical sub dataframe, we'll cut it into bins.

In [ ]:
# Add Fare to sub dataframes
df_con["Fare"] = train["Fare"]
df_bin["Fare"] = pd.cut(
    train["Fare"], bins=5
)  # discretised because the feature had continuous float values

In [ ]:
df_bin.Fare.value_counts()

In [ ]:
df_con.Fare.value_counts()

In [ ]:
# Visualise the Fare bin counts as well as the Fare distribution versus Survived.
plot_count_dist(
    data=train,
    bin_df=df_bin,
    label_column="Survived",
    target_column="Fare",
    figsize=(20, 10),
    use_bin_df=True,
)

### Feature: Cabin

In [ ]:
# how many missing values do we have in this column
train.Cabin.isnull().sum()

Since there are too many missing values, we won't use Cabin for our initial models and won't add it to our sub dataframes.

### Feature: Embarked

In [ ]:
# How many missing values in this column
train.Embarked.isnull().sum()

In [ ]:
# What kind of values are there in Embarked
train.Embarked.value_counts()

In [ ]:
# Visualize the counts
sns.countplot(y="Embarked", data=train)

In [ ]:
# For now let's remove the rows which don't have an Embarked value
df_bin["Embarked"] = train["Embarked"]
df_con["Embarked"] = train["Embarked"]

# Remove Embarked rows which are missing values
print(len(df_con))
df_con = df_con.dropna(subset=["Embarked"])
df_bin = df_bin.dropna(subset=["Embarked"])
print(len(df_con))

## Feature Encoding

Let's first check our two subset dataframes


In [ ]:
df_bin.head()  # discretized or binned

In [ ]:
df_con.head()  # continuous

In [ ]:
# One-hot encode binned variables
one_hot_cols = df_bin.columns.tolist()
one_hot_cols.remove("Survived")
df_bin_enc = pd.get_dummies(df_bin, columns=one_hot_cols)

df_bin_enc.head()

In [ ]:
# One hot encode the categorical columns
df_embarked_one_hot = pd.get_dummies(df_con["Embarked"], prefix="embarked")

df_sex_one_hot = pd.get_dummies(df_con["Sex"], prefix="sex")

df_plcass_one_hot = pd.get_dummies(df_con["Pclass"], prefix="pclass")

In [ ]:
# Combine the one hot encoded columns with df_con_enc
df_con_enc = pd.concat([df_con, df_embarked_one_hot, df_sex_one_hot, df_plcass_one_hot], axis=1)

# Drop the original categorical columns (because now they've been one hot encoded)
df_con_enc = df_con_enc.drop(["Pclass", "Sex", "Embarked"], axis=1)

In [ ]:
# Let's look at df_con_enc
df_con_enc.head(20)

## Machine Learning Models

In [ ]:
# Seclect the dataframe we want to use first for predictions
selected_df = df_con_enc

In [ ]:
selected_df.head()

In [ ]:
# Split the dataframe into data and labels
X_train = selected_df.drop("Survived", axis=1)  # data
y_train = selected_df.Survived  # labels

In [ ]:
# Shape of the data (without labels)
X_train.shape

In [ ]:
X_train.head()

In [ ]:
# Shape of the labels
y_train.shape

In [ ]:
# Function that runs the requested algorithm and returns the accuracy metrics
from experiment_kaggle_titanic_competition_submission import fit_ml_algo

### Logistic Regression


In [ ]:
# Logistic Regression
start_time = time.time()
train_pred_log, acc_log, acc_cv_log, model_log = fit_ml_algo(
    LogisticRegression(), X_train, y_train, 10
)
log_time = time.time() - start_time
print("Accuracy: %s" % acc_log)
print("Accuracy CV 10-Fold: %s" % acc_cv_log)
print("Running Time: %s" % datetime.timedelta(seconds=log_time))

### k-NN

In [ ]:
# k-Nearest Neighbours
start_time = time.time()
train_pred_knn, acc_knn, acc_cv_knn, model_knn = fit_ml_algo(
    KNeighborsClassifier(n_neighbors=3), X_train, y_train, 10
)
knn_time = time.time() - start_time
print("Accuracy: %s" % acc_knn)
print("Accuracy CV 10-Fold: %s" % acc_cv_knn)
print("Running Time: %s" % datetime.timedelta(seconds=knn_time))

### Gaussian Naive Bayes

In [ ]:
# Gaussian Naive Bayes
start_time = time.time()
train_pred_gaussian, acc_gaussian, acc_cv_gaussian, model_gaussian = fit_ml_algo(
    GaussianNB(), X_train, y_train, 10
)
gaussian_time = time.time() - start_time
print("Accuracy: %s" % acc_gaussian)
print("Accuracy CV 10-Fold: %s" % acc_cv_gaussian)
print("Running Time: %s" % datetime.timedelta(seconds=gaussian_time))

### Linear Support Vector Classifier

In [ ]:
# Linear SVC
start_time = time.time()
train_pred_svc, acc_linear_svc, acc_cv_linear_svc, model_linear_svc = fit_ml_algo(
    LinearSVC(), X_train, y_train, 10
)
linear_svc_time = time.time() - start_time
print("Accuracy: %s" % acc_linear_svc)
print("Accuracy CV 10-Fold: %s" % acc_cv_linear_svc)
print("Running Time: %s" % datetime.timedelta(seconds=linear_svc_time))

### Stochastic Gradient Descent Classifier

In [ ]:
# Stochastic Gradient Descent
start_time = time.time()
train_pred_sgd, acc_sgd, acc_cv_sgd, model_sgd = fit_ml_algo(SGDClassifier(), X_train, y_train, 10)
sgd_time = time.time() - start_time
print("Accuracy: %s" % acc_sgd)
print("Accuracy CV 10-Fold: %s" % acc_cv_sgd)
print("Running Time: %s" % datetime.timedelta(seconds=sgd_time))

### Decision Tree Classifier

In [ ]:
# Decision Tree Classifier
start_time = time.time()
train_pred_dt, acc_dt, acc_cv_dt, model_dt = fit_ml_algo(
    DecisionTreeClassifier(), X_train, y_train, 10
)
dt_time = time.time() - start_time
print("Accuracy: %s" % acc_dt)
print("Accuracy CV 10-Fold: %s" % acc_cv_dt)
print("Running Time: %s" % datetime.timedelta(seconds=dt_time))

### Gradient Boosting Trees

In [ ]:
# Gradient Boosting Trees
start_time = time.time()
train_pred_gbt, acc_gbt, acc_cv_gbt, model_gbt = fit_ml_algo(
    GradientBoostingClassifier(), X_train, y_train, 10
)
gbt_time = time.time() - start_time
print("Accuracy: %s" % acc_gbt)
print("Accuracy CV 10-Fold: %s" % acc_cv_gbt)
print("Running Time: %s" % datetime.timedelta(seconds=gbt_time))

### Random Forest Classifier

In [ ]:
from sklearn.ensemble import RandomForestClassifier

start_time = time.time()
train_pred_rf, acc_rf, acc_cv_rf, model_rf = fit_ml_algo(
    RandomForestClassifier(n_estimators=50), X_train, y_train, 10
)
rf_time = time.time() - start_time
print("Accuracy: %s" % acc_rf)
print("Accuracy CV 10-Fold: %s" % acc_cv_rf)
print("Running Time: %s" % datetime.timedelta(seconds=rf_time))

### Voting Classifier

In [ ]:
from sklearn.ensemble import VotingClassifier

start_time = time.time()
train_pred_vt, acc_vt, acc_cv_vt, model_vt = fit_ml_algo(
    VotingClassifier(
        estimators=[
            ("rf", model_rf),
            ("gbt", model_gbt),
            ("dt", model_dt),
            ("knn", model_knn),
        ],
        voting="soft",
    ),
    X_train,
    y_train,
    10,
)
vt_time = time.time() - start_time
print("Accuracy: %s" % acc_vt)
print("Accuracy CV 10-Fold: %s" % acc_cv_vt)
print("Running Time: %s" % datetime.timedelta(seconds=vt_time))

### Catboost Algorithm

In [ ]:
pass  # Environment setup moved to README; use local inputs.

In [ ]:
from catboost import CatBoostClassifier, Pool, cv

In [ ]:
# View the data for the CatBoost model
X_train.head()

In [ ]:
# View the labels for the CatBoost model
y_train.head()

In [ ]:
# Define the categorical features for the CatBoost model
cat_features = np.where(X_train.dtypes != float)[0]
cat_features

In [ ]:
# Use the CatBoost Pool() function to pool together the training data and categorical feature labels
train_pool = Pool(X_train, y_train, cat_features)

In [ ]:
# CatBoost model definition
catboost_model = CatBoostClassifier(
    iterations=1000, custom_loss=["Accuracy"], loss_function="Logloss"
)

# Fit CatBoost model
catboost_model.fit(train_pool, plot=True)

# CatBoost accuracy
acc_catboost = round(catboost_model.score(X_train, y_train) * 100, 2)

In [ ]:
# How long will this take?
start_time = time.time()

# Set params for cross-validation as same as initial model
cv_params = catboost_model.get_params()

# Run the cross-validation for 10-folds (same as the other models)
cv_data = cv(train_pool, cv_params, fold_count=10, plot=True)

# How long did it take?
catboost_time = time.time() - start_time

# CatBoost CV results save into a dataframe (cv_data), let's withdraw the maximum accuracy score
acc_cv_catboost = round(np.max(cv_data["test-Accuracy-mean"]) * 100, 2)

In [ ]:
# Print out the CatBoost model metrics
print("---CatBoost Metrics---")
print("Accuracy: {}".format(acc_catboost))
print("Accuracy cross-validation 10-Fold: {}".format(acc_cv_catboost))
print("Running Time: {}".format(datetime.timedelta(seconds=catboost_time)))

## Model Results

In [ ]:
models = pd.DataFrame(
    {
        "Model": [
            "KNN",
            "Logistic Regression",
            "Naive Bayes",
            "Stochastic Gradient Decent",
            "Linear SVC",
            "Decision Tree",
            "Gradient Boosting Trees",
            "Random Forests",
            "Voting Classifier",
            "CatBoost",
        ],
        "Score": [
            acc_knn,
            acc_log,
            acc_gaussian,
            acc_sgd,
            acc_linear_svc,
            acc_dt,
            acc_gbt,
            acc_rf,
            acc_vt,
            acc_catboost,
        ],
    }
)
print("---Reuglar Accuracy Scores---")
models.sort_values(by="Score", ascending=False)

In [ ]:
cv_models = pd.DataFrame(
    {
        "Model": [
            "KNN",
            "Logistic Regression",
            "Naive Bayes",
            "Stochastic Gradient Decent",
            "Linear SVC",
            "Decision Tree",
            "Gradient Boosting Trees",
            "Random Forest",
            "Voting Classifier",
            "CatBoost",
        ],
        "Score": [
            acc_cv_knn,
            acc_cv_log,
            acc_cv_gaussian,
            acc_cv_sgd,
            acc_cv_linear_svc,
            acc_cv_dt,
            acc_cv_gbt,
            acc_cv_rf,
            acc_cv_vt,
            acc_cv_catboost,
        ],
    }
)
print("---Cross-validation Accuracy Scores---")
cv_models.sort_values(by="Score", ascending=False)

## Feature Importance


Which features were most important for making predictions?

In [ ]:
# Feature Importance
from experiment_kaggle_titanic_competition_submission import feature_importance
# plt.savefig('catboost_feature_importance.png')

In [ ]:
# Plot the feature importance scores
feature_importance(catboost_model, X_train)

## Precision, Recall, F1, and AUC Scores

In [ ]:
metrics = ["Precision", "Recall", "F1", "AUC"]

eval_metrics = catboost_model.eval_metrics(train_pool, metrics=metrics, plot=True)

for metric in metrics:
    print(str(metric) + ": {}".format(np.mean(eval_metrics[metric])))

## Submission

In [ ]:
# Our test dataframe has some columns our model hasn't been trained on
test.head()

In [ ]:
# One hot encode the columns in the test data frame (like X_train)
test_embarked_one_hot = pd.get_dummies(test["Embarked"], prefix="embarked")

test_sex_one_hot = pd.get_dummies(test["Sex"], prefix="sex")

test_plcass_one_hot = pd.get_dummies(test["Pclass"], prefix="pclass")

In [ ]:
# Combine the test one hot encoded columns with test
test = pd.concat([test, test_embarked_one_hot, test_sex_one_hot, test_plcass_one_hot], axis=1)

In [ ]:
# Let's look at test, it should have one hot encoded columns now
test.head()

We converted the columns in the test set to the same format as df_con_enc

In [ ]:
# Create a list of columns to be used for the predictions
wanted_test_columns = X_train.columns
wanted_test_columns

In [ ]:
# Make a prediction using the CatBoost model on the wanted columns
predictions = catboost_model.predict(test[wanted_test_columns])

In [ ]:
# Create a submisison dataframe and append the relevant columns
submission = pd.DataFrame()
submission["PassengerId"] = test["PassengerId"]
submission["Survived"] = predictions  # our model predictions on the test dataset
submission.head()

In [ ]:
# What does our submission have to look like?
gender_submission.head()

In [ ]:
# Let's convert our submission dataframe 'Survived' column to ints
submission["Survived"] = submission["Survived"].astype(int)
print("Converted Survived column to integers.")

In [ ]:
# How does our submission dataframe look?
submission.head()

In [ ]:
# Are our test and submission dataframes the same length?
if len(submission) == len(test):
    print("Submission dataframe is the same length as test ({} rows).".format(len(submission)))
else:
    print("Dataframes mismatched, won't be able to submit to Kaggle.")

In [ ]:
# Convert submisison dataframe to csv for submission to csv
# for Kaggle submisison
submission.to_csv(INPUT_PATHS["CATBOOST_SUBMISSION_CSV"], index=False)
print("Submission CSV is ready!")